In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#example
PROMPT_TEMPLATE = "At the {year} Academy Awards, Best Actor winner {answer} gave the most celebrated performance of the year."

correct_answers = {
    1999: "Benigni",
    2000: "Spacey",
    2001: "Crowe",
    2002: "Washington",
    2003: "Brody",
    2004: "Penn",
    2005: "Foxx",
    2006: "Hoffman",
    2007: "Whitaker",
    2008: "Day-Lewis",
    2009: "Penn",
    2010: "Bridges",
    2011: "Firth",
    2012: "Dujardin",
    2013: "Day-Lewis",
    2014: "McConaughey",
    2015: "Redmayne",
    2016: "DiCaprio",
    2017: "Affleck",
    2018: "Oldman",
    2019: "Malek",
    2020: "Phoenix",
    2021: "Hopkins",
    2022: "Smith",
    2023: "Fraser",
    2024: "Murphy",
}

years = sorted(correct_answers.keys())



def build_wrong_year_map(years, mode="future_shift", shift=5):
    """
    mode:
      - 'future_shift': wrong year = year + shift if possible, else year - shift
      - 'past_shift':   wrong year = year - shift if possible, else year + shift
      - 'furthest':     wrong year = far end of range
    """
    years = sorted(years)
    min_year, max_year = min(years), max(years)

    wrong_year_map = {}

    for y in years:
        if mode == "future_shift":
            candidate = y + shift
            if candidate <= max_year:
                wrong_year_map[y] = candidate
            else:
                wrong_year_map[y] = y - shift

        elif mode == "past_shift":
            candidate = y - shift
            if candidate >= min_year:
                wrong_year_map[y] = candidate
            else:
                wrong_year_map[y] = y + shift

        elif mode == "furthest":
            dist_to_min = abs(y - min_year)
            dist_to_max = abs(y - max_year)
            wrong_year_map[y] = min_year if dist_to_min < dist_to_max else max_year

        else:
            raise ValueError("mode must be one of: future_shift, past_shift, furthest")

    return wrong_year_map


WRONG_YEAR_MAP = build_wrong_year_map(years, mode="future_shift", shift=5)



def safe_model_name(model_name: str):
    return model_name.split("/")[-1].replace(".", "_")


def answer_to_token_ids(answer_text, tokenizer):
    enc = tokenizer(answer_text, add_special_tokens=False, return_attention_mask=False)
    token_ids = enc["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return token_ids, tokens


@torch.no_grad()
def pseudo_log_likelihood_score(text, tokenizer, model, device="cpu", max_length=128):
    """
    Standard pseudo-log-likelihood:
    mask each token one at a time, score the true token, average log-prob.
    """
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"][0]
    attention_mask = enc["attention_mask"][0]

    special_ids = set(tokenizer.all_special_ids)

    token_logps = []
    tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist())

    for pos in range(len(input_ids)):
        true_id = input_ids[pos].item()

        # Skip special tokens
        if true_id in special_ids:
            continue

        masked_ids = input_ids.clone()
        masked_ids[pos] = tokenizer.mask_token_id

        outputs = model(
            input_ids=masked_ids.unsqueeze(0).to(device),
            attention_mask=attention_mask.unsqueeze(0).to(device),
        )

        logits = outputs.logits[0, pos]
        log_probs = torch.log_softmax(logits, dim=-1)
        token_logps.append(log_probs[true_id].item())

    if len(token_logps) == 0:
        raise ValueError("No non-special tokens were scored.")

    mean_logp = float(np.mean(token_logps))
    sum_logp = float(np.sum(token_logps))

    return {
        "mean_logp": mean_logp,
        "sum_logp": sum_logp,
        "n_scored_tokens": len(token_logps),
        "pseudo_prob": float(np.exp(mean_logp)),
        "tokens": tokens,
    }


def build_temporal_pair_df(prompt_template, correct_answers, wrong_year_map):
    rows = []

    for year, answer in correct_answers.items():
        wrong_year = wrong_year_map[year]

        correct_text = prompt_template.format(year=year, answer=answer)
        wrong_text = prompt_template.format(year=wrong_year, answer=answer)

        rows.append({
            "Correct_Year": year,
            "Wrong_Year": wrong_year,
            "Year_Gap": wrong_year - year,
            "Answer": answer,
            "Correct_Text": correct_text,
            "Wrong_Text": wrong_text,
        })

    return pd.DataFrame(rows).sort_values("Correct_Year").reset_index(drop=True)


pair_df = build_temporal_pair_df(PROMPT_TEMPLATE, correct_answers, WRONG_YEAR_MAP)
pair_df.head()

In [ ]:
def run_temporal_discrimination_eval(
    model_name,
    pair_df,
    device="cpu",
    max_length=128,
):
    model_tag = safe_model_name(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.to(device)
    model.eval()

    rows = []

    for _, row in tqdm(pair_df.iterrows(), total=len(pair_df), desc=f"Scoring {model_tag}"):
        correct_out = pseudo_log_likelihood_score(
            text=row["Correct_Text"],
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length,
        )

        wrong_out = pseudo_log_likelihood_score(
            text=row["Wrong_Text"],
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_length=max_length,
        )

        margin_mean = correct_out["mean_logp"] - wrong_out["mean_logp"]
        margin_sum = correct_out["sum_logp"] - wrong_out["sum_logp"]

        rows.append({
            "Correct_Year": int(row["Correct_Year"]),
            "Wrong_Year": int(row["Wrong_Year"]),
            "Year_Gap": int(row["Year_Gap"]),
            "Answer": row["Answer"],
            "Correct_Text": row["Correct_Text"],
            "Wrong_Text": row["Wrong_Text"],
            "Correct_MeanLogP": correct_out["mean_logp"],
            "Wrong_MeanLogP": wrong_out["mean_logp"],
            "Correct_PseudoProb": correct_out["pseudo_prob"],
            "Wrong_PseudoProb": wrong_out["pseudo_prob"],
            "Margin_MeanLogP": margin_mean,   # primary
            "Margin_SumLogP": margin_sum,     # robustness
            "Prefers_Correct": margin_mean > 0,
        })

    results_df = pd.DataFrame(rows).sort_values("Correct_Year").reset_index(drop=True)

    # Plot 1: primary metric
    plt.figure(figsize=(12, 5))
    plt.bar(results_df["Correct_Year"], results_df["Margin_MeanLogP"])
    plt.axhline(0, linestyle="--")
    plt.xlabel("Correct year")
    plt.ylabel("Margin: correct - wrong (mean log-prob)")
    plt.title(f"Robustness check by year: {model_tag}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Plot 2: robustness check
    plt.figure(figsize=(12, 5))
    plt.bar(results_df["Correct_Year"], results_df["Margin_SumLogP"])
    plt.axhline(0, linestyle="--")
    plt.xlabel("Correct year")
    plt.ylabel("Margin: correct - wrong (sum log-prob)")
    plt.title(f"Robustness check by year: {model_tag}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame([{
        "model": model_name,
        "mean_margin_meanlogp": results_df["Margin_MeanLogP"].mean(),
        "mean_margin_sumlogp": results_df["Margin_SumLogP"].mean(),
        "fraction_prefers_correct": results_df["Prefers_Correct"].mean(),
        "n_years": len(results_df),
    }])

    del tokenizer
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results_df, summary

In [ ]:
MODEL_NAME = "manelalab/chrono-bert-v1-20241231"

results_df, summary_df = run_temporal_discrimination_eval(
    model_name=MODEL_NAME,
    pair_df=pair_df,
    device=DEVICE,
    max_length=128,
)

summary_df

In [ ]:
MODEL_NAMES = [
    "manelalab/chrono-bert-v1-20001231",
    "manelalab/chrono-bert-v1-20101231",
    "manelalab/chrono-bert-v1-20151231",
    "manelalab/chrono-bert-v1-20201231",
]

all_results = {}
all_summaries = []

for model_name in MODEL_NAMES:
    results_df, summary_df = run_temporal_discrimination_eval(
        model_name=model_name,
        pair_df=pair_df,
        device=DEVICE,
        max_length=128,
    )
    all_results[model_name] = results_df
    all_summaries.append(summary_df)

all_summary_df = pd.concat(all_summaries, ignore_index=True)
all_summary_df

In [ ]:
def plot_combined_temporal_discrimination(
    all_results,
    metric="Margin_SumLogP",   # use plot 2
    title=None,
):
    plt.figure(figsize=(11, 5.5))

    pretty_names = {
        "manelalab/chrono-bert-v1-20001231": "ChronoBERT-2000",
        "manelalab/chrono-bert-v1-20101231": "ChronoBERT-2010",
        "manelalab/chrono-bert-v1-20151231": "ChronoBERT-2015",
        "manelalab/chrono-bert-v1-20201231": "ChronoBERT-2020",
    }

    ordered_models = [
        "manelalab/chrono-bert-v1-20001231",
        "manelalab/chrono-bert-v1-20101231",
        "manelalab/chrono-bert-v1-20151231",
        "manelalab/chrono-bert-v1-20201231",
    ]

    for model_name in ordered_models:
        df = all_results[model_name].sort_values("Correct_Year")
        plt.plot(
            df["Correct_Year"],
            df[metric],
            linewidth=2,
            label=pretty_names.get(model_name, safe_model_name(model_name)),
        )

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("Correct year")
    plt.ylabel("Margin: correct - wrong (sum log-prob)")

    if title is None:
        title = "Temporal discrimination margin by year"
    plt.title(title)

    plt.grid(True, axis="y", alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# %%
plot_combined_temporal_discrimination(
    all_results=all_results,
    metric="Margin_SumLogP",
    title="",
)